# 🖥️ Guia Completo: Terminal, Scripts e Ambientes Virtuais no Colab

Para abrir um terminal no Google Colab, siga estas instruções:

1. **Usando o menu Comandos (na linha abaixo dos Menus)**
    *   Clique em `Comandos`.
    *   Na caixa de seleção que aparece, digite `Mostrar terminal`.
    * Abrirá um painel na parte lateral da sua tela com um shell de terminal onde você pode executar comandos Linux.

2.  **Melhorando a visualização do painel**
    * Clique nos três pontinhos (encima à direita no painel).
    * Escolha "Alterar layout da página" -> "Visualização de uma guia".
    * As guias (Notebook, Terminal, Gemini, etc.) ficam agrupadas no painel.
    * Para separar novamente as guias, escolha "Duas colunas" ou "Duas linhas" para o layout da página.

## Executando scripts Python no Colab

Este notebook permite rodar scripts `.py` que estão em `/content` sem precisar abrir o terminal.

Use as células abaixo para executar `system_health_check.py` diretamente no runtime do Colab.

Se o arquivo não estiver na pasta `/content`, lembre-se de ajustar o caminho da variável antes de executar.

### Execução rápida com %run

Use esta versão mais curta e direta para executar o script no runtime atual do notebook. Isso é equivalente a rodar o código diretamente aqui.

In [3]:
%run /content/system_health_check.py

--- Disk Health for . ---
Usage: 9.13%
Free Space: 205 GB

--- Files larger than 10MB ---


### Execução detalhada com subprocess

Use esta versão avançada quando precisar capturar a saída padrão (`stdout`) e os erros (`stderr`) separadamente, o que é ótimo para lidar com logs e depuração.

In [4]:
script_path = "/content/system_health_check.py"

import os
import subprocess
import sys

if os.path.exists(script_path):
    print(f"Running: {script_path}")
    result = subprocess.run(
        [sys.executable, script_path],
        capture_output=True,
        text=True
    )
    print(result.stdout)
    if result.stderr:
        print("--- stderr ---")
        print(result.stderr)
else:
    print(f"File not found: {script_path}")

Running: /content/system_health_check.py
--- Disk Health for . ---
Usage: 9.13%
Free Space: 205 GB

--- Files larger than 10MB ---



## Criar um Virtualenv em /content via Bash

Esta etapa cria um ambiente virtual isolado chamado `myenv` na pasta `/content` e instala as dependências listadas no arquivo `requirements.txt`.

Isso é uma ótima prática no Colab para evitar conflitos de versão com as bibliotecas que o Google já pré-instala globalmente.

**💡 Nota sobre `venv` vs `virtualenv`:**
O script abaixo tenta primeiro usar o módulo nativo do Python (`python3 -m venv`). No entanto, como o ambiente base do Colab às vezes restringe pacotes do sistema necessários (gerando erros com o `ensurepip`), o script possui um "plano B" inteligente. Se o `venv` falhar, ele automaticamente instala e utiliza a biblioteca `virtualenv` (`python3 -m virtualenv`), que é mais robusta e garante a criação do ambiente isolado sem depender das configurações do sistema operacional.

In [5]:
%%bash
set -euo pipefail

cd /content

if [[ ! -f requirements.txt ]]; then
  echo "ERROR: /content/requirements.txt not found."
  echo "Upload requirements.txt to /content and run again."
  exit 1
fi

# Clean up a broken previous env if it exists
rm -rf /content/myenv

# Colab can fail with python -m venv (ensurepip). Use virtualenv fallback.
if python3 -m venv /content/myenv; then
  echo "Created env with python -m venv"
else
  echo "python -m venv failed; switching to virtualenv..."
  python3 -m pip install --quiet virtualenv
  python3 -m virtualenv /content/myenv
fi

source /content/myenv/bin/activate
python -m pip install --upgrade pip
python -m pip install -r /content/requirements.txt

echo "Virtual environment ready: /content/myenv"
python -m pip --version

python -m venv failed; switching to virtualenv...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.0/469.0 kB 26.3 MB/s eta 0:00:00
created virtual environment CPython3.12.13.final.0-64-x86_64 in 550ms
  creator CPython3Posix(dest=/content/myenv, clear=False, no_vcs_ignore=False, global=False)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.cache/virtualenv)
    added seed packages: pip==26.0.1
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 35.3 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.0.1
    Uninstalling pip-26.0.1:
      Successfully uninstalled pip-26.0.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 93.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 627.3/627.3 kB 24.1 MB/s  0:00:00
   ━━━━━━━━━━━━━

Error: Command '['/content/myenv/bin/python3', '-m', 'ensurepip', '--upgrade', '--default-pip']' returned non-zero exit status 1.


In [6]:
%%bash
set -euo pipefail

/content/myenv/bin/python -c "import sys; import pandas as pd; print(sys.executable); print(pd.__version__)"

/content/myenv/bin/python
3.0.2


In [7]:
%%bash
set -euo pipefail

if [[ ! -x /content/myenv/bin/python ]]; then
  echo "ERROR: /content/myenv/bin/python not found."
  echo "Run the virtualenv setup cell first."
  exit 1
fi

if [[ ! -f /content/system_health_check.py ]]; then
  echo "ERROR: /content/system_health_check.py not found."
  echo "Upload the script to /content and run again."
  exit 1
fi

/content/myenv/bin/python /content/system_health_check.py

--- Disk Health for . ---
Usage: 9.25%
Free Space: 204 GB

--- Files larger than 10MB ---


## 💡 Entendendo o uso de Ambientes Virtuais e Pip no Colab

### O que é o `-m` no comando `python -m pip`?
A flag `-m` significa **"module"** (módulo). Ele diz ao Python para procurar o módulo especificado (como o `pip`) e executá-lo como se fosse o script principal. Isso garante que o pacote será instalado para o interpretador Python exato que está rodando o comando, evitando conflitos de ambiente.

### Python Global vs. Python do Virtualenv
* **`python -m pip`**: Usa o Python padrão do sistema (no Colab, o ambiente global). Pacotes instalados assim afetam o notebook inteiro, mas podem causar conflitos com as versões pré-instaladas pelo Google.
* **`/content/myenv/bin/python -m pip`**: Utiliza o caminho absoluto do interpretador do seu ambiente virtual. Instala as dependências de forma segura e isolada na pasta `/content/myenv/`.

### Por que não usar `source activate` ou `%run`?
* **`source /content/myenv/bin/activate`**: O Colab executa comandos `!` ou `%%bash` em um *subshell* (terminal temporário). O ambiente é ativado, mas **se perde imediatamente** quando a célula termina de rodar.
* **`%run`**: Executa scripts utilizando o ambiente global atual da interface do notebook, ignorando o ambiente virtual criado em background pelo bash.

### A Solução: Como usar o Virtualenv nas células normais do Colab
Para importar bibliotecas instaladas no seu `myenv` diretamente nas células de código Python, a melhor abordagem é adicionar o diretório de pacotes do seu ambiente virtual ao caminho do Python (`sys.path`), como mostrado na célula abaixo.

In [8]:
import sys
import glob

# Procura automaticamente a pasta site-packages do ambiente virtual
venv_lib_path = glob.glob('/content/myenv/lib/python*/site-packages')

if venv_lib_path:
    venv_site_packages = venv_lib_path[0]
    # Insere o caminho na primeira posição do sys.path para dar prioridade
    if venv_site_packages not in sys.path:
        sys.path.insert(0, venv_site_packages)
        print(f"✅ Caminho adicionado: {venv_site_packages}")
    else:
        print("Caminho do ambiente virtual já está no sys.path.")
else:
    print("❌ Pasta site-packages do ambiente virtual não encontrada. Você criou o myenv?")

# Após rodar esta célula, você poderá fazer os imports normalmente:
# import pacote_instalado_no_myenv


✅ Caminho adicionado: /content/myenv/lib/python3.12/site-packages


### Exemplo Prático de Importação

Agora que o caminho do ambiente virtual foi adicionado ao `sys.path`, você pode importar as bibliotecas instaladas nele diretamente nas células do Colab.

Vamos testar importando o `pandas` e verificando de qual diretório ele está sendo carregado para confirmar que é a versão do nosso `myenv`.

In [9]:
try:
    import pandas as pd
    print(f"✅ Pandas importado com sucesso!")
    print(f"📌 Versão: {pd.__version__}")
    print(f"📂 Caminho do módulo: {pd.__file__}")
except ImportError:
    print("❌ Erro: O pacote 'pandas' não foi encontrado. Certifique-se de que ele está listado no requirements.txt e que o ambiente foi criado.")

✅ Pandas importado com sucesso!
📌 Versão: 3.0.2
📂 Caminho do módulo: /content/myenv/lib/python3.12/site-packages/pandas/__init__.py


## 🌍 Ambientes Virtuais em Diferentes Contextos: Colab vs. Nuvem vs. Local

É importante entender como o comportamento dos ambientes virtuais muda dependendo de **onde** seu código está sendo executado:

### 1. Colab Hosted Runtime (Ambiente Padrão do Google)
*   **Características:** É um contêiner efêmero (temporário) que já possui centenas de bibliotecas de Data Science pré-instaladas globalmente.
*   **Uso:** Criar um virtualenv aqui (como fizemos ao longo deste notebook) é uma técnica de **isolamento temporário** para evitar conflitos de versão com pacotes nativos do Google.
*   **Atenção:** Quando o runtime é desconectado, a pasta `/content` é apagada e o ambiente é perdido (a menos que seja criado dentro do Google Drive, montado em `/content/drive`).

### 2. Google Cloud (Vertex AI Workbench ou Compute Engine)
*   **Características:** Máquinas virtuais dedicadas com armazenamento **persistente**.
*   **Uso:** O uso de virtualenvs (via `conda`, `venv` ou `poetry`) ou contêineres Docker é o padrão da indústria e altamente recomendado. O ambiente permanece intacto mesmo após a máquina ser desligada, sendo o caminho certo para projetos duradouros e produção.

### 3. Ambiente Local e VS Code
*   **Características:** Execução no seu próprio hardware, onde você tem controle e responsabilidade total sobre o sistema operacional.
*   **Uso:** O uso de virtualenvs é praticamente **obrigatório** no Python para não quebrar dependências globais do seu computador (evitando o temido *Dependency Hell*).
*   **💡 Uso da Extensão Colab no VS Code:** Se você estiver utilizando a extensão do Google Colab no VS Code para usufruir da interface local enquanto roda o código na nuvem, é aconselhável seguir rigorosamente o procedimento descrito no [User Guide Oficial](https://github.com/googlecolab/colab-vscode/wiki/User-Guide). Preste atenção especial à etapa de **selecionar um kernel do Colab para executar o notebook**, garantindo que seus scripts rodem no ambiente (e virtualenv) correto configurado lá na ponta do servidor.

## 💾 Persistindo o Virtualenv no Google Drive

Como vimos, a pasta `/content` é apagada quando o runtime do Colab é encerrado. Para manter seu ambiente virtual (e os pacotes instalados) salvos permanentemente, você pode criá-lo dentro do seu Google Drive.

### 1. Montar o Google Drive
Primeiro, precisamos conectar o seu Google Drive ao Colab. O código abaixo pedirá sua permissão para acessar os arquivos.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 2. Criar e Instalar Pacotes no Drive
Agora, adaptamos o nosso script bash para criar o ambiente em `/content/drive/MyDrive/colab_env` (ou qualquer outra pasta da sua escolha).

*Nota: Operações de disco no Google Drive são mais lentas que no armazenamento local efêmero, então a instalação inicial pode demorar um pouco mais, mas você só precisará fazê-la uma vez!*

In [ ]:
%%bash
set -euo pipefail

# Define o caminho do ambiente no Google Drive
DRIVE_ENV_PATH="/content/drive/MyDrive/colab_env"

# Se o ambiente já existir, não precisamos recriar do zero
if [[ -d "$DRIVE_ENV_PATH" ]]; then
  echo "O ambiente virtual já existe em: $DRIVE_ENV_PATH"
else
  echo "Criando novo ambiente virtual no Google Drive..."
  python3 -m pip install --quiet virtualenv
  python3 -m virtualenv "$DRIVE_ENV_PATH"
fi

# Ativa o ambiente e instala/atualiza dependências (exemplo com pandas)
source "$DRIVE_ENV_PATH/bin/activate"
python -m pip install --upgrade pip
# python -m pip install -r /content/requirements.txt  # Descomente se tiver um requirements.txt
python -m pip install pandas --quiet

echo "Ambiente no Drive pronto e atualizado!"


### 3. Adicionar o Caminho do Drive ao `sys.path`
Assim como fizemos antes, para usar os pacotes instalados neste ambiente persistente nas células normais do Colab, precisamos apontar o Python para a pasta `site-packages` do seu Drive.

In [ ]:
import sys
import glob

# Procura a pasta site-packages do ambiente no Drive
drive_venv_path = glob.glob('/content/drive/MyDrive/colab_env/lib/python*/site-packages')

if drive_venv_path:
    drive_site_packages = drive_venv_path[0]
    if drive_site_packages not in sys.path:
        sys.path.insert(0, drive_site_packages)
        print(f"✅ Caminho do Drive adicionado: {drive_site_packages}")
    else:
        print("Caminho do ambiente no Drive já está no sys.path.")
else:
    print("❌ Pasta site-packages não encontrada no Drive.")

# Testando a importação
try:
    import pandas as pd
    print(f"📌 Pandas importado de: {pd.__file__}")
except ImportError:
    print("❌ Erro ao importar pandas.")


### 4. Reutilizando o Ambiente em Novas Sessões

Quando você iniciar uma nova sessão no Colab (um novo ambiente de execução) no futuro, você **não** precisará instalar todos os pacotes novamente! Como eles já estão salvos permanentemente no seu Google Drive, basta montar o Drive e apontar o caminho do Python (`sys.path`) para a pasta existente.

Você pode usar o bloco de código abaixo como a primeira célula do seu notebook sempre que abrir uma nova sessão:

In [ ]:
import sys
import glob
from google.colab import drive

# 1. Conecta o Google Drive à sessão atual
drive.mount('/content/drive')

# 2. Localiza e adiciona a pasta do ambiente já salvo ao sys.path do Colab
drive_venv_path = glob.glob('/content/drive/MyDrive/colab_env/lib/python*/site-packages')

if drive_venv_path:
    drive_site_packages = drive_venv_path[0]
    if drive_site_packages not in sys.path:
        sys.path.insert(0, drive_site_packages)
        print(f"✅ Ambiente virtual do Drive carregado com sucesso!\nCaminho: {drive_site_packages}")
    else:
        print("Caminho do ambiente no Drive já está configurado no sys.path.")
else:
    print("❌ Ambiente não encontrado no Drive. Verifique se o caminho está correto ou se a instalação inicial foi feita.")


### 5. Verificando a Origem da Importação

Para confirmar se o Colab está realmente usando os pacotes do seu ambiente virtual no Drive, você pode inspecionar o atributo `__file__` do pacote importado. Ele revela o caminho absoluto do arquivo no sistema.

In [ ]:
try:
    import pandas as pd
    print(f"📦 Versão do Pandas: {pd.__version__}")
    print(f"📂 Caminho da instalação: {pd.__file__}\n")

    # Verifica se o caminho aponta para o Google Drive
    if 'drive/MyDrive/colab_env' in pd.__file__:
        print("✅ Sucesso! O pacote está sendo carregado do seu ambiente virtual no Google Drive.")
    else:
        print("❌ Atenção: O pacote está sendo carregado do ambiente padrão do Colab.")

except ImportError:
    print("❌ Erro ao importar pandas. Verifique se a instalação foi concluída com sucesso no Drive.")